# 4a — Cell quality
Takes output from 1-Preprocessing_MEA2.ipynb

Decide, once per experiment, which cells are good enough for the analyses that follow, and
save the verdicts in one file: `<output_directory>/<exp>_cell_quality.pkl`.

Three criteria, each optional and each remembered separately:

| step | criterion | needs |
|------|-----------|-------|
| 1 | **RPV** — refractory-period violations below a threshold | the phy sorting output |
| 2 | **STA** — a well-defined receptive field (you review the STA figures) | notebook 2 (checkerboard) |
| 3 | **chirp** — a clear chirp response (you review the chirp rasters) — *optional* | a chirp recording |

The result is a plain dictionary, one entry per cell (like the STA results):

```python
cell_quality[cell] = {"rpv": 0.31, "rpv_ok": True, "sta_ok": True, "chirp_ok": None}
```

`None` means that step was not run for that cell. Any later notebook then filters its cells
with it:

```python
cell_quality = utils.load_cell_quality(params)
good = utils.good_cells(cell_quality, cells)                        # every criterion evaluated here
good = utils.good_cells(cell_quality, cells, ["rpv_ok", "sta_ok"])  # or choose the criteria
good = [c for c in cells if cell_quality[c]["rpv_ok"]]              # or by hand
```

Re-running this notebook **keeps** earlier verdicts: each step only overwrites its own
criterion, and only when you run it.

In [5]:
%load_ext autoreload
%autoreload 2

import os
import matplotlib.pyplot as plt

import params_thijs as params
import utils
from utils import chirp as analysis

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Step 0: Load the cells

All the sorted cells of the experiment (from `<exp>_fullexp_neurons_data.pkl`). If a quality
file already exists for this experiment it is loaded, so you can redo one step without losing
the others.

In [8]:
loadname = params.output_directory / f'{params.experiment}_fullexp_neurons_data.pkl'
if loadname.exists():

    spike_trains = utils.load_obj(
        params.output_directory / f'{params.experiment}_fullexp_neurons_data.pkl'
    )
    cells = list(spike_trains.keys())
    print(f"{len(cells)} cells in this experiment")

    # Reuse the saved quality if there is one (earlier verdicts are kept), else start empty.
    try:
        cell_quality = utils.load_cell_quality(params)
        print("Loaded the existing cell quality:")
        utils.describe_cell_quality(cell_quality)
    except FileNotFoundError:
        cell_quality = utils.new_cell_quality(cells)
        print("No cell quality saved yet for this experiment: starting from scratch.")

FileNotFoundError: [Errno 2] No such file or directory: 'C:/thijs/sono_data/2026-09-09 mouse c57 758 Mekano6 A/processed/standard_analysis_pipeline/2026-09-09 mouse c57 758 Mekano6 A_fullexp_neurons_data.pkl'

## Step 1: Refractory-period violations (RPV)

Two spikes closer than the refractory period cannot come from one neuron, so a high RPV rate
means a contaminated cluster. Cells with RPV below `rpv_threshold` are flagged good.
Requires the phy sorting output (`params.phy_directory`).

In [ ]:
rpv_len = 2  # ms: inter-spike intervals below this count as refractory violations
rpv_threshold = 0.5  # %: maximum acceptable RPV rate

cell_rpvs = utils.get_cell_rpvs(cells, params.phy_directory, rpv_len)
for cell in cells:
    cell_quality[cell]["rpv"] = cell_rpvs[cell]["rpv"]
    cell_quality[cell]["rpv_ok"] = cell_rpvs[cell]["rpv"] < rpv_threshold
utils.save_cell_quality(cell_quality, params)

rpv_good = utils.good_cells(cell_quality, cells, ["rpv_ok"])
print(f"{len(rpv_good)}/{len(cells)} cells pass the RPV filter (RPV < {rpv_threshold}%)")

# Distribution of RPV rates, with the threshold.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([cell_quality[c]["rpv"] for c in cells], bins=40, color="#3AA6B9")
ax.axvline(rpv_threshold, color="red", ls="--", label=f"threshold = {rpv_threshold}%")
ax.set_xlabel("RPV rate (%)", fontsize=14)
ax.set_ylabel("number of cells", fontsize=14)
ax.legend(fontsize=12)
plt.show()

## Step 2: Cells with a good STA

Keep the cells that have a **well-defined receptive field**. Each cell's checkerboard STA
figure (notebook 2) is shown one at a time — type `y` to keep it, Enter to reject it. Only
the cells not rejected by the RPV step are reviewed; the others stay `None` for `sta_ok`
(and `None` never passes).

**Every answer is saved immediately.** You can stop at any time (interrupt the kernel, close
the notebook...) and re-run this cell later: it continues with the cells you have not
answered yet. Set `redo_sta_review = True` to forget the STA answers and start over.

In [ ]:
redo_sta_review = False

check_directory = utils.find_analysis_directory(params.output_directory, "Checkerboard")

# Candidates = cells not rejected by the RPV step (all cells if it was not run).
sta_candidates = [c for c in cells if cell_quality[c]["rpv_ok"] is not False]
if redo_sta_review:
    for cell in sta_candidates:
        cell_quality[cell]["sta_ok"] = None

# Resume: only the candidates without an answer yet are shown.
to_review = [c for c in sta_candidates if cell_quality[c]["sta_ok"] is None]
print(f"{len(sta_candidates) - len(to_review)} already answered, {len(to_review)} to review")


def save_sta_decision(cell, kept):
    cell_quality[cell]["sta_ok"] = kept
    utils.save_cell_quality(cell_quality, params, verbose=False)


utils.review_cells_by_sta(to_review, check_directory, on_decision=save_sta_decision)
print(f"{len(utils.good_cells(cell_quality, cells, ['sta_ok']))} cells with a good STA")

## Step 3 (optional): Cells with a good chirp

Skip this section if there is no chirp recording. Otherwise it computes and plots the chirp
rasters (with the spatial STA next to them), then you review them — type `y` to keep the
cells with a **clear chirp response**. The rasters are saved for notebook 4b (cell typing),
which needs them.

In [ ]:
old = False  # False = new 50 Hz chirp ; True = old 2p-room chirp

(
    chirp_cells,
    spike_times,
    stim_onsets,
    vec_keys,
    check_directory,
    CT_directory,
    old,
    chirp_recording,
) = analysis.get_all_inputs_for_chirp_analysis(params, old)

cell_data = analysis.compute_chirp_rasters(
    chirp_cells, spike_times, stim_onsets, vec_keys, old=old, n_bins=800, n_bins_small=16000
)
analysis.save_chirp_rasters(cell_data, old, chirp_recording, CT_directory, params)

### Plot the chirp rasters

One figure per cell, saved in `CT_directory/Chirp_rasters+STA/`. Figures already on disk are
kept unless you say otherwise (plotting is the slow part). Requires the checkerboard analysis
for the STA panel.

In [ ]:
analysis.plot_chirp_rasters(chirp_cells, cell_data, CT_directory, check_directory, old=old)

### Review the chirp rasters

Only the cells not rejected by the RPV and STA steps are reviewed. As for the STA, **every
answer is saved immediately**: re-running this cell continues with the cells not answered
yet; set `redo_chirp_review = True` to start the chirp review over.

In [ ]:
redo_chirp_review = False

# Candidates = cells not rejected by the RPV or STA steps.
chirp_candidates = [
    c
    for c in cells
    if cell_quality[c]["rpv_ok"] is not False and cell_quality[c]["sta_ok"] is not False
]
if redo_chirp_review:
    for cell in chirp_candidates:
        cell_quality[cell]["chirp_ok"] = None

# Resume: only the candidates without an answer yet are shown.
to_review = [c for c in chirp_candidates if cell_quality[c]["chirp_ok"] is None]
print(f"{len(chirp_candidates) - len(to_review)} already answered, {len(to_review)} to review")


def save_chirp_decision(cell, kept):
    cell_quality[cell]["chirp_ok"] = kept
    utils.save_cell_quality(cell_quality, params, verbose=False)


utils.review_cells_by_chirp(to_review, CT_directory, on_decision=save_chirp_decision)
print(f"{len(utils.good_cells(cell_quality, cells, ['chirp_ok']))} cells with a good chirp")

## Step 4 (optional): Correct a verdict by hand

`cell_quality` is an ordinary Python dictionary: one entry per cell, and inside it one
value per criterion. To change a verdict, assign it directly and save:

```python
cell_quality[208]            # -> {"rpv": 0.31, "rpv_ok": True, "sta_ok": False, "chirp_ok": None}
cell_quality[208]["sta_ok"] = True      # the review was wrong: this STA is fine
cell_quality[45]["chirp_ok"] = False    # on second thought, reject this chirp
cell_quality[12]["sta_ok"] = None       # "not reviewed" (it will be asked again on re-run)
utils.save_cell_quality(cell_quality, params)
```

Allowed values for `rpv_ok`, `sta_ok`, `chirp_ok`: `True` (good), `False` (bad), `None` (not
evaluated — never passes). `rpv` is the measured rate in %, for information. The cell below
is a place for such corrections; it ends with the summary of what every other notebook will
get from `utils.good_cells(cell_quality, cells)`.

In [ ]:
# Look at one cell:
# print(cell_quality[208])

# Correct verdicts here, e.g.:
# cell_quality[208]["sta_ok"] = True
# cell_quality[45]["chirp_ok"] = False

utils.save_cell_quality(cell_quality, params)
good_cells = utils.describe_cell_quality(cell_quality)